# 3 · Modeling and simulation

This stage generates the pool of host–guest **configurations** that are screened
energetically in the next notebook. It does *not* compute any binding energy —
both methods below are configuration generators only (see the box in
[`modeling-and-simulation.md`](../modeling-and-simulation/modeling-and-simulation.md#the-sampling-problem)).

Two complementary strategies are used (methodology **§5**):

1. **Stochastic rigid-body docking (SRD)** — unbiased random placement of the
   guest in the pore (`051_SRD`), via a modified
   [`host_guest`](https://github.com/bafgreat/host_guest) package.
2. **Classical molecular dynamics (MD)** — force-field-guided sampling in LAMMPS
   (`052_MD`), which first requires building force-field topology files
   (EQeq host charges, RESP guest charges, GAFF2 typing).

> **Heavy external tooling.** This notebook orchestrates several external
> programs (`host_guest`, EQeq, ORCA, Multiwfn, AmberTools, `lammps_interface`,
> LAMMPS), run partly on a workstation and partly on an HPC cluster. The cells
> here **generate inputs and post-process outputs**; the heavy calculations run
> separately. Reusable file-format helpers come from
> [`mof-guest-toolkit`](https://github.com/adricu12/mof-guest-toolkit)
> (`cif_tools`, `lammps`); see [`REQUIREMENTS.md`](../REQUIREMENTS.md) for every
> dependency and version. Cells are committed without execution outputs.

---
## 5.1 Stochastic rigid-body docking (SRD)

SRD generation runs on the modified
[`host_guest`](https://github.com/bafgreat/host_guest) package (install steps in
the table below). The next two cells build a **self-contained** `051_SRD/` folder:
the optimized host CIFs and guest XYZ files are copied in, and one SLURM job is
written per host–guest pair and seed by filling the
[`submit-hpc.sh`](./templates/submit-hpc.sh) template with the host, guest, and
random seed from [`data/simulation-seeds.csv`](../data/simulation-seeds.csv)
(three independent seeds per pair). The whole folder can then be copied to the
cluster on its own — no other part of the repository is needed there.

In [ ]:
import pandas as pd
from pathlib import Path
import shutil

In [ ]:
# ----------------------------
# Paths
# ----------------------------

slurm_job_template = Path("./templates/submit-hpc.sh")

if not slurm_job_template.exists():
    raise FileNotFoundError(f"Template not found: {slurm_job_template.resolve()}")

seed_file = Path("../data/simulation-seeds.csv")

if not seed_file.exists():
    raise FileNotFoundError(f"Seed file not found: {seed_file.resolve()}")

df = pd.read_csv(seed_file)

hosts = sorted(df["host"].unique().tolist())
guests = sorted(df["guest_abbr"].unique().tolist())
srd_runs = ["srd_seed1", "srd_seed2", "srd_seed3"]

# Main SRD folder
srd_root = Path("../thesis/051_SRD")

# Structure folders inside 051_SRD
host_structure_folder = srd_root / "hosts_cif"
guests_structure_folder = srd_root / "guests_xyz"

host_structure_folder.mkdir(parents=True, exist_ok=True)
guests_structure_folder.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Generate files
# ----------------------------

for h in hosts:

    # Original host optimized CIF
    h_source_cif = Path(f"../thesis/011_HOSTS/final_structures/{h}_dftb_geo-opt.cif")

    if not h_source_cif.exists():
        print(f"No CIF file found for host {h}: {h_source_cif}")
        continue

    # Copy host CIF into 051_SRD/hosts_cif
    h_target_cif = host_structure_folder / h_source_cif.name
    shutil.copy2(h_source_cif, h_target_cif)

    # Path relative to each host folder inside 051_SRD/<host>/
    h_workflow_cif = Path("../hosts_cif") / h_source_cif.name

    # Create host SRD folder
    srd_host_folder = srd_root / h
    srd_host_folder.mkdir(parents=True, exist_ok=True)

    for g in guests:

        # Original guest optimized XYZ
        g_source_xyz = Path(f"../thesis/012_GUESTS/final_structures/{g}.xyz")

        if not g_source_xyz.exists():
            print(f"No XYZ file found for guest {g}: {g_source_xyz}")
            continue

        # Copy guest XYZ into 051_SRD/guests_xyz
        g_target_xyz = guests_structure_folder / g_source_xyz.name
        shutil.copy2(g_source_xyz, g_target_xyz)

        # Path relative to each host folder inside 051_SRD/<host>/
        g_workflow_xyz = Path("../guests_xyz") / g_source_xyz.name

        for s in srd_runs:

            match = df.loc[
                (df["host"] == h) & (df["guest_abbr"] == g),
                s
            ]

            if match.empty:
                print(f"No seed found for host={h}, guest={g}, run={s}")
                continue

            seed = int(match.iloc[0])

            slurm_job_file = srd_host_folder / f"{h}_{g}_{s}_{seed}.sh"

            # Copy template
            shutil.copy2(slurm_job_template, slurm_job_file)

            # Replace placeholders
            text = slurm_job_file.read_text()

            text = text.replace("job-id", f"{h}_{g}_{s}")
            text = text.replace("HOST", str(h_workflow_cif))
            text = text.replace("GUEST", str(g_workflow_xyz))
            text = text.replace("SEED", str(seed))

            slurm_job_file.write_text(text)

            print(f"Created: {slurm_job_file}")

Copy the `thesis/051_SRD/` folder to your HPC workspace — it is self-contained,
so the rest of the `thesis/` tree is not needed there.

Before submitting, install the modified
[`host_guest`](https://github.com/bafgreat/host_guest) package on the HPC. The
modifications (user-defined random seeds and HDF5 output) are in
[`scripts/helpers/host_guest_repo_update/`](./helpers/host_guest_repo_update/);
copy each file into a clone of `host_guest` at the location below:

| This repo | Place in `host_guest` at |
|---|---|
| [`docker.py`](./helpers/host_guest_repo_update/docker.py) | `host_guest/energy/docker.py` |
| [`generate_complexes.py`](./helpers/host_guest_repo_update/generate_complexes.py) | `host_guest/setter/generate_complexes.py` |
| [`hdf5_library.py`](./helpers/host_guest_repo_update/hdf5_library.py) | `host_guest/io/hdf5_library.py` |
| [`test_hdf5_library.py`](./helpers/host_guest_repo_update/test_hdf5_library.py) | `host_guest/tests/test_hdf5_library.py` |

Each SRD job then runs, per host–guest pair and seed:

```bash
complexes_from_file HOST_CIF GUEST_XYZ -nc 20000 -s SEED
```

## 5.2 Molecular dynamics sampling (MD)

The MD route samples configurations in LAMMPS, which first needs force-field
**topology files** for every host and guest (built under
`052_MD/0521_topo_file_gen/`). This is the most involved part of the pipeline: it
chains several external tools, and each step below produces one piece of the
force field. §5.2.1 builds the host topology (EQeq charges + UFF4MOF) and the
guest topology (RESP charges + GAFF2); §5.2.3 then assembles and launches the
trajectories. Only the final `0522_MD_trajectories/` folder is copied to the HPC.

In [ ]:
from pathlib import Path
import re
import shutil
from typing import List, Tuple

from mof_toolkit import (
    read_structure, gen_inp_file,        # ORCA/AMS input generation
    simplify_cif, inject_eqeq_charges,   # EQeq CIF pre/post-formatting (cif_tools)
    center_guest_in_box,                 # place a guest in the host box (lammps)
)

# Stage 5.2 (MD) topology and input files live under 052_MD.
md_root = Path("../thesis/052_MD")
host_structure_folder = md_root / "0521_topo_file_gen" / "hosts"
guests_structure_folder = md_root / "0521_topo_file_gen" / "guests"
host_structure_folder.mkdir(parents=True, exist_ok=True)
guests_structure_folder.mkdir(parents=True, exist_ok=True)

### 5.2.1 Host topology files (EQeq charges)

The host framework is described by UFF4MOF parameters (assigned by
`lammps_interface`) plus partial charges from the **EQeq** method. EQeq requires
a specifically formatted CIF, so each optimized host CIF is first simplified to a
minimal P1 cell, charged with EQeq, and the charges injected back for
`lammps_interface`.

**Step 1 — simplify the CIF.** `simplify_cif` (from `mof_toolkit.cif_tools`)
rewrites each optimized host CIF as the minimal P1 cell the EQeq C++ code expects.

In [ ]:
hosts_cif_folder = Path("../thesis/011_HOSTS/final_structures")

hosts_cif_files = sorted(hosts_cif_folder.glob("*.cif"))

for h in hosts_cif_files:
    outfile = host_structure_folder / f"{h.stem}_formatted.cif"
    simplify_cif(h, outfile)

**Step 2 — assign EQeq charges.** Compile the
[EQeq C++ code](./helpers/cpp_EQeq/) once, then run it on each simplified host
CIF. The binary must run from the folder holding the CIFs (or be on your `PATH`).
For every host it writes a charged CIF named
`<host>_formatted.cif_EQeq_Ewald_1.20_-2.00.cif`, plus `.mol` and `.pdb` files.

In [ ]:
%%bash
cd ../thesis/052_MD/0521_topo_file_gen/hosts

# Compile
g++ EQeq_v1_00.cpp -o eqeq

for file in ./*.cif; do
    echo "Running EQeq on $file"
    ./eqeq "$file"
done

**Step 3 — reformat the charged CIF.** EQeq writes the partial charges in a
separate, reordered atom loop. `inject_eqeq_charges` matches each charge back onto
the reference structure (per element, under periodic boundary conditions) and
appends an `_atom_site_charge` column — the layout `lammps_interface` expects.

In [ ]:
# Inject the EQeq partial charges back onto the reference host CIFs so that
# lammps_interface can read them (cif_tools.inject_eqeq_charges matches atoms
# per element under PBC).
charged_dir = md_root / "0521_topo_file_gen" / "hosts"        # EQeq output CIFs
reference_dir = Path("../thesis/011_HOSTS/final_structures")  # original detailed CIFs
output_dir = md_root / "0521_topo_file_gen" / "hosts_eqeq"    # charged CIFs for lammps_interface
output_dir.mkdir(parents=True, exist_ok=True)

# Exact suffix produced by the EQeq C++ code.
charged_suffix = "_formatted.cif_EQeq_Ewald_1.20_-2.00.cif"

for charged_cif in sorted(charged_dir.glob(f"*{charged_suffix}")):
    mof = charged_cif.name.removesuffix(charged_suffix)
    reference_cif = reference_dir / f"{mof}.cif"
    if not reference_cif.exists():
        print(f"Missing reference CIF for {mof}: {reference_cif}")
        continue
    inject_eqeq_charges(reference_cif, charged_cif, output_dir / f"{mof}_EQeq.cif")
    print(f"  {mof} -> {mof}_EQeq.cif")

**Step 4 — write the host LAMMPS data file.**
[`lammps_interface`](https://github.com/peteboyd/lammps_interface) assigns the
UFF4MOF force field and converts each charged host CIF into a LAMMPS `data.*`
file. The next three cells (1) define the `lammps_interface` run parameters
(2×2×2 supercell, 12.5 Å cutoff, fixed metal centres), (2) wrap the call so each
CIF is processed inside its own folder, and (3) run it over every charged CIF in
`hosts_eqeq/`. Run these with the `lammps_interface` environment active.

In [ ]:
from pathlib import Path
import os
import traceback

from lammps_interface.structure_data import from_CIF, write_CIF
from lammps_interface.lammps_main import LammpsSimulation

In [ ]:
class Parameters:
    def __init__(self, cif):
        self.cif_file = str(cif)
        self.output_cif = False
        self.output_raspa = False

        self.force_field = "UFF4MOF"
        self.mol_ff = None
        self.h_bonding = False
        self.dreid_bond_type = "harmonic"
        self.fix_metal = True
        
        self.minimize = True
        self.bulk_moduli = False
        self.thermal_scaling = False
        self.npt = False
        self.nvt = True
        self.cutoff = 12.5
        self.replication = "2,2,2"
        self.orthogonalize = False
        self.random_vel = True
        self.dump_dcd = 0
        self.dump_xyz = 0
        self.dump_lammpstrj = 100
        self.restart = False
        
        self.tol = 0.4
        self.neighbour_size = 5
        self.iter_count = 10
        self.max_dev = 0.01
        self.temp = 298.0
        self.pressure = 1.0
        self.nprodstp = 200000
        self.neqstp = 200000
        
        self.insert_molecule = None
        self.deposit = 0

In [ ]:
def generate_lammps_data_in_same_folder(cif_file):
    cif_file = Path(cif_file).resolve()
    workdir = cif_file.parent
    original_cwd = Path.cwd()

    try:
        os.chdir(workdir)

        par = Parameters(cif_file.name)  # use filename because cwd is now hosts_eqeq

        sim = LammpsSimulation(par)
        cell, graph = from_CIF(par.cif_file)

        sim.set_cell(cell)
        sim.set_graph(graph)
        sim.split_graph()
        sim.assign_force_fields()
        sim.compute_simulation_size()
        sim.merge_graphs()
        sim.write_lammps_files()

        print(f"Generated files for: {cif_file.name}")

    finally:
        os.chdir(original_cwd)

In [ ]:
hosts_eqeq_folder = Path("../thesis/052_MD/0521_topo_file_gen/hosts_eqeq").resolve()

cif_files = sorted(hosts_eqeq_folder.glob("*.cif"))

print(f"Found {len(cif_files)} CIF files")

for cif_file in cif_files:
    print(f"\n=== Processing {cif_file.name} ===")

    try:
        generate_lammps_data_in_same_folder(cif_file)

    except Exception as error:
        print(f"Failed for {cif_file.name}: {error}")
        traceback.print_exc()

This will output the data files in 052_MD/0521_topo_file_gen/hosts_eqeq 

### 5.2.2 Guest topology files (RESP charges + GAFF2)

Guest molecules are parameterized with **GAFF2** and **RESP** charges derived
from an HF/6-31G(d) wavefunction. The five-step pipeline is: ORCA single point →
Multiwfn RESP fit → Antechamber/parmchk2 (GAFF2 typing) → tLEaP topology →
`amber2lammps` (LAMMPS `data.` file).

**Step 1 — ORCA single point.** RESP charges are fitted to an HF/6-31G(d)
wavefunction. The cell below writes one ORCA input per guest from the
[`spe_orca.inp`](./templates/spe_orca.inp) template, using the optimized guest
geometry (via the toolkit's `gen_inp_file`). Outputs go to
`052_MD/0521_topo_file_gen/guests/<guest>/`.

In [ ]:
from mof_toolkit import read_structure, gen_inp_file
from pathlib import Path

In [ ]:
# Input folder: optimized guest XYZ files
guests_geo_opt_xyz_folder = Path("../thesis/012_GUESTS/final_structures").resolve()

# Output folder: ORCA input files for MD/topology workflow
guests_output_root = Path("../thesis/052_MD/0521_topo_file_gen/guests").resolve()
guests_output_root.mkdir(parents=True, exist_ok=True)

# Template
template_file = Path("./templates/spe_orca.inp").resolve()

if not template_file.exists():
    raise FileNotFoundError(f"Template not found: {template_file}")

# Gather guest geo-opt xyz files 
xyz_files = sorted(guests_geo_opt_xyz_folder.glob("*.xyz"))

print(f"Found {len(xyz_files)} xyz files")

for guest_xyz in xyz_files:
    guest_name = guest_xyz.stem

    guest_dir = guests_output_root / guest_name
    guest_dir.mkdir(parents=True, exist_ok=True)

    structure = read_structure(guest_xyz)

    file_name = f"{guest_name}_spe_resp"
    output_file = guest_dir / f"{file_name}.inp"

    gen_inp_file(
        template_file,
        *structure,
        name=file_name,
        output=output_file,
    )

    print(f"Created: {output_file}")

These orca files can be ran using the local computer, although the entire thesis/052_MD/0521_topo_file_gen/guests folder can be copied and submitted using slurm jobs. 

Once the ORCA single points are done, Multiwfn needs a **Molden** file, generated
from each ORCA output with `orca_2mkl <guest> -molden`.

In [ ]:
%%bash

for folder in ../thesis/052_MD/0521_topo_file_gen/guests/*; do
    if [ -d "$folder" ]; then
        echo "Entering $folder"

        for inp_file in "$folder"/*.out; do
            # If no .inp exists, skip
            [ -e "$inp_file" ] || continue

            # Get filename without path and without .inp
            name=$(basename "$inp_file" .out)

            echo "Running orca_2mkl for $name"

            # Run inside the folder so output files are written there
            (
                cd "$folder"
                orca_2mkl "$name" -molden
            )
        done
    fi
done

**Step 2 — RESP charge fitting (Multiwfn).** The ORCA single points produce a
`.molden.input` per guest. Because Multiwfn is run as a Windows executable here
(see [`REQUIREMENTS.md`](../REQUIREMENTS.md)), the files are staged into one
folder, processed on Windows, and the resulting charges collected back:

1. Collect all `.molden.input` files from `052_MD/0521_topo_file_gen/guests/*/`.
2. Copy them into a staging folder `multiwfn_molden_inputs/`.
3. Move that folder to the Windows Multiwfn directory.
4. Run the Windows batch script (below) to fit RESP charges.
5. Copy the generated `.chg` files back to the matching guest folders.

In [ ]:
from pathlib import Path
import shutil

# Source: guest folders generated in the topology workflow
guests_root = Path("../thesis/052_MD/0521_topo_file_gen/guests").resolve()

# Staging folder: all .molden.input files together
staging_root = Path("../thesis/052_MD/0521_topo_file_gen/multiwfn_molden_inputs").resolve()
staging_root.mkdir(parents=True, exist_ok=True)

molden_files = sorted(guests_root.glob("*/*.molden.input"))

print(f"Found {len(molden_files)} Molden input files")

for molden_file in molden_files:
    target_file = staging_root / molden_file.name

    # Avoid overwriting if two files have the same name
    if target_file.exists():
        target_file = staging_root / f"{molden_file.parent.name}_{molden_file.name}"

    shutil.copy2(molden_file, target_file)

    print(f"Copied: {molden_file} -> {target_file}")

../thesis/052_MD/0521_topo_file_gen/multiwfn_molden_inputs/ \
├── AC_spe_resp.molden \
├── IBU_spe_resp.molden \
├── NAP_spe_resp.molden \
│   

### Manual Windows step

Copy the folder

`thesis/052_MD/0521_topo_file_gen/multiwfn_molden_inputs`

and the file

[`data/resp_multiwfn_commands.txt`](../data/resp_multiwfn_commands.txt)

to the Windows Multiwfn working directory, for example

`C:\Users\adri\Desktop\Multiwfn\multiwfn_molden_inputs`

keeping the same structure: one subfolder per guest, each with one
`.molden.input` file.

@echo off
setlocal enabledelayedexpansion

rem Paths
set "BASEDIR=C:\Users\adria\Desktop\Multiwfn\multiwfn_molden_inputs"
set "MULTIWFNEXE=C:\Users\adria\Desktop\Multiwfn\Multiwfn.exe"
set "RESPINPUT=C:\Users\adria\Desktop\Multiwfn\resp_multiwfn_commands.txt"

rem Go to folder with all .molden.input files
cd /d "%BASEDIR%"

rem Run Multiwfn for every .molden.input file
for %%M in (*.molden.input) do (
    echo Processing %%M ...
    "%MULTIWFNEXE%" "%%M" < "%RESPINPUT%"
    echo Finished %%M
    echo.
)

echo All done.
endlocal

Run the batch file inside the Windows Multiwfn directory, then copy the resulting
`.chg` files back into `multiwfn_molden_inputs/`. Antechamber only needs the
charge values, so the next cell strips each `.chg` to its last column and copies
it into the matching guest folder.

In [ ]:
from pathlib import Path

# Folder containing raw Multiwfn .chg files
chg_root = Path("../thesis/052_MD/0521_topo_file_gen/multiwfn_molden_inputs").resolve()

# Folder containing guest folders: NAP, AC, KETO, etc.
guests_root = Path("../thesis/052_MD/0521_topo_file_gen/guests").resolve()

chg_files = sorted(chg_root.glob("*.chg"))

print(f"Found {len(chg_files)} .chg files")

for chg_file in chg_files:
    # Example: NAP_spe_resp.chg
    file_name = chg_file.stem          # NAP_spe_resp
    guest_name = file_name.replace("_spe_resp", "")  # NAP

    guest_dir = guests_root / guest_name

    if not guest_dir.exists():
        print(f"Missing guest folder for {guest_name}: {guest_dir}")
        continue

    output_chg = guest_dir / chg_file.name

    # Keep only the last column
    charges = []
    with chg_file.open("r") as f:
        for line in f:
            parts = line.split()
            if parts:
                charges.append(parts[-1] + "\n")

    output_chg.write_text("".join(charges))

    print(f"Written: {output_chg}")

**Step 3 — GAFF2 atom typing (AmberTools 25).** Run with the AmberTools environment
active. `antechamber` reads the ORCA output directly (`-fi orcout`), assigns GAFF2
atom types, and applies the external RESP charges (`-c rc -cf …`); `parmchk2` then
generates any missing force-field parameters into a `.frcmod` file.

In [ ]:
from pathlib import Path
import subprocess

guests_root = Path("../thesis/052_MD/0521_topo_file_gen/guests").resolve()

# Find only charge files inside direct guest subfolders
chg_files = sorted(guests_root.glob("*/*.chg"))

print(f"Found {len(chg_files)} .chg files")

for chg_file in chg_files:
    guest_dir = chg_file.parent

    # Example: NAP_spe_resp.chg -> NAP_spe_resp
    file_name = chg_file.stem

    out_file = guest_dir / f"{file_name}.out"
    mol2_file = guest_dir / f"{file_name}_leap.mol2"
    frcmod_file = guest_dir / f"{file_name}.frcmod"

    if not out_file.exists():
        print(f"Skipping {guest_dir.name}: missing {out_file.name}")
        continue

    print(f"\nProcessing {guest_dir.name}")
    print(f"Using charge file: {chg_file.name}")

    antechamber_command = [
        "antechamber",
        "-i", out_file.name,
        "-fi", "orcout",
        "-o", mol2_file.name,
        "-fo", "mol2",
        "-c", "rc",
        "-cf", chg_file.name,
        "-at", "gaff2",
        "-nc", "0",
    ]

    parmchk2_command = [
        "parmchk2",
        "-i", mol2_file.name,
        "-f", "mol2",
        "-o", frcmod_file.name,
        "-s", "gaff2",
    ]

    try:
        subprocess.run(antechamber_command, cwd=guest_dir, check=True)
        print(f"Created: {mol2_file.name}")

        subprocess.run(parmchk2_command, cwd=guest_dir, check=True)
        print(f"Created: {frcmod_file.name}")

    except subprocess.CalledProcessError as e:
        print(f"Failed for {guest_dir.name}: {e}")

**Step 4 — topology assembly (tLEaP).** First write a `leap.in` per guest from the
[`leap.in`](./templates/leap.in) template (substituting the guest file name); the
next cell then runs tLEaP.

In [ ]:
from pathlib import Path

# Folder containing guest folders
guests_root = Path("../thesis/052_MD/0521_topo_file_gen/guests").resolve()

# tleap template
leap_template = Path("./templates/leap.in").resolve()

if not leap_template.exists():
    raise FileNotFoundError(f"Leap template not found: {leap_template}")

guest_dirs = sorted([p for p in guests_root.iterdir() if p.is_dir()])

print(f"Found {len(guest_dirs)} guest folders")

for guest_dir in guest_dirs:
    guest_name = guest_dir.name
    file_name = f"{guest_name}_spe_resp"

    # Optional check: only make leap.in if the charge file exists
    chg_file = guest_dir / f"{file_name}.chg"
    if not chg_file.exists():
        print(f"Skipping {guest_name}: missing {chg_file.name}")
        continue

    leap_output = guest_dir / "leap.in"

    text = leap_template.read_text()
    text = text.replace("<file_name>", file_name)

    leap_output.write_text(text)

    print(f"Written: {leap_output}")

Then run tLEaP over every guest folder. It produces the Amber topology
(`.prmtop`) and coordinate (`.inpcrd`) files.

In [ ]:
from pathlib import Path
import subprocess

guests_root = Path("../thesis/052_MD/0521_topo_file_gen/guests").resolve()

guest_dirs = sorted([p for p in guests_root.iterdir() if p.is_dir()])

print(f"Found {len(guest_dirs)} guest folders")

for guest_dir in guest_dirs:
    guest_name = guest_dir.name
    file_name = f"{guest_name}_spe_resp"

    frcmod_file = guest_dir / f"{file_name}.frcmod"
    mol2_file = guest_dir / f"{file_name}_leap.mol2"
    leap_file = guest_dir / "leap.in"

    # These will be created by tleap, so we check them after running
    prmtop_file = guest_dir / f"{file_name}.prmtop"
    inpcrd_file = guest_dir / f"{file_name}.inpcrd"

    missing_inputs = []

    if not frcmod_file.exists():
        missing_inputs.append(frcmod_file.name)

    if not mol2_file.exists():
        missing_inputs.append(mol2_file.name)

    if not leap_file.exists():
        missing_inputs.append(leap_file.name)

    if missing_inputs:
        print(f"Skipping {guest_name}: missing {', '.join(missing_inputs)}")
        continue

    print(f"\nRunning tleap for {guest_name}")

    command = ["tleap", "-s", "-f", "leap.in"]

    try:
        subprocess.run(command, cwd=guest_dir, check=True)

        if prmtop_file.exists() and inpcrd_file.exists():
            print(f"Created: {prmtop_file.name}")
            print(f"Created: {inpcrd_file.name}")
        else:
            print(f"WARNING: tleap finished, but expected output files were not found for {guest_name}")
            if not prmtop_file.exists():
                print(f"  Missing: {prmtop_file.name}")
            if not inpcrd_file.exists():
                print(f"  Missing: {inpcrd_file.name}")

    except subprocess.CalledProcessError as e:
        print(f"tleap failed for {guest_name}: {e}")

**Step 5 — convert to LAMMPS format.** `amber2lammps.py` reads the Amber topology
(`.top`) and coordinates (`.crd`) and writes a LAMMPS `data.<guest>` file. It runs
under **Python 2** (see [`REQUIREMENTS.md`](../REQUIREMENTS.md)). Rename the tLEaP
outputs `.prmtop`→`.top` and `.inpcrd`→`.crd` before running this cell.

In [ ]:
from pathlib import Path
import subprocess

guests_root = Path("../thesis/052_MD/0521_topo_file_gen/guests").resolve()

# Central location of amber2lammps.py
amber2lammps_script = Path("./helpers/amber2lammps.py").resolve()

# Python 2 executable from your environment
python2_exe = "python2"


if not amber2lammps_script.exists():
    raise FileNotFoundError(f"amber2lammps.py not found: {amber2lammps_script}")

guest_dirs = sorted([p for p in guests_root.iterdir() if p.is_dir()])

print(f"Found {len(guest_dirs)} guest folders")

for guest_dir in guest_dirs:
    guest_name = guest_dir.name
    file_name = f"{guest_name}_spe_resp"

    top_file = guest_dir / f"{file_name}.top"
    crd_file = guest_dir / f"{file_name}.crd"

    if not top_file.exists():
        print(f"Skipping {guest_name}: missing {top_file.name}")
        continue

    if not crd_file.exists():
        print(f"Skipping {guest_name}: missing {crd_file.name}")
        continue

    print(f"\nProcessing {guest_name}")

    command = [
        python2_exe,
        str(amber2lammps_script),
        file_name,
    ]

    try:
        subprocess.run(command, cwd=guest_dir, check=True)

        data_file = guest_dir / f"data.{file_name}"

        if data_file.exists():
            print(f"Created: {data_file.name}")
        else:
            print(f"WARNING: expected data file not found for {guest_name}")

    except subprocess.CalledProcessError as e:
        print(f"amber2lammps failed for {guest_name}: {e}")

Finally, gather every guest `data.*` file into a single `guests_resp/` folder, so
the MD-setup step below can pair each guest with each host.

In [ ]:
from pathlib import Path
import shutil

# Folder containing guest subfolders
guests_root = Path("../thesis/052_MD/0521_topo_file_gen/guests").resolve()

# New folder to collect all LAMMPS data files
guests_resp_root = Path("../thesis/052_MD/0521_topo_file_gen/guests_resp").resolve()
guests_resp_root.mkdir(parents=True, exist_ok=True)

# Find data files inside direct guest subfolders
data_files = sorted(guests_root.glob("*/data.*"))

print(f"Found {len(data_files)} data files")

for data_file in data_files:
    target_file = guests_resp_root / data_file.name

    # Avoid overwriting if two data files somehow have the same name
    if target_file.exists():
        target_file = guests_resp_root / f"{data_file.parent.name}_{data_file.name}"

    shutil.copy2(data_file, target_file)

    print(f"Copied: {data_file} -> {target_file}")

print("Done.")

### 5.2.3 LAMMPS MD setup

With host and guest topology files ready, the trajectories are assembled under
`052_MD/0522_MD_trajectories/<host>/<guest>/`. For each host–guest pair the guest
is **centered in the host simulation box** (one cell of the 2×2×2 supercell) with
`center_guest_in_box`, producing a `data.<guest>_centered` file that LAMMPS later
appends to the host via `read_data ... add append`.

In [ ]:
from pathlib import Path
import re, math
import numpy as np
from typing import List, Tuple
import shutil

seed_file = Path("../data/simulation-seeds.csv")

In [ ]:
# Center each guest in each host box, producing data.<guest>_centered files
# ready to be combined with the host via LAMMPS `read_data ... add append`.
hosts_eqeq_dir = (md_root / "0521_topo_file_gen" / "hosts_eqeq").resolve()
guests_resp_dir = (md_root / "0521_topo_file_gen" / "guests_resp").resolve()
md_traj_root = (md_root / "0522_MD_trajectories").resolve()
md_traj_root.mkdir(parents=True, exist_ok=True)

host_data_files = sorted(hosts_eqeq_dir.glob("data.*"))
guest_data_files = sorted(guests_resp_dir.glob("data.*"))
print(f"Found {len(host_data_files)} host and {len(guest_data_files)} guest data files")

# data.NU-902_..._EQeq -> NU-902 ; data.AC_spe_resp -> AC
host_name = lambda f: f.name.removeprefix("data.").split("_")[0]
guest_name = lambda f: f.name.removeprefix("data.").replace("_spe_resp", "")

for host_data_file in host_data_files:
    host_out_dir = md_traj_root / host_name(host_data_file)
    host_out_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(host_data_file, host_out_dir / host_data_file.name)
    print(f"\n[HOST] {host_name(host_data_file)}")

    for guest_data_file in guest_data_files:
        try:
            out_path, _ = center_guest_in_box(
                guest_data_file, host_data_file,
                out_root=host_out_dir,
                guest_name=guest_name(guest_data_file),
                replication=(2, 2, 2),
            )
            print(f"  [OK] {guest_name(guest_data_file)} -> {out_path.name}")
        except Exception as exc:
            print(f"  [ERROR] {host_name(host_data_file)}/{guest_name(guest_data_file)}: {exc}")

**Generate the LAMMPS input files.** For every host–guest pair, one input is
written from the [`in.lammps_complex-gen`](./templates/in.lammps_complex-gen)
template for each of the seven trajectories — three at 300 K with distinct seeds,
plus one each at 600, 900, 1200, and 1500 K — filling in the target temperature
and random seed. Each run gets its own `<T>K_seed<…>/` subfolder, completing the
directory tree ready for HPC submission.

In [ ]:
from pathlib import Path
import pandas as pd
import re

# =========================================================
# Paths
# =========================================================

md_root = Path("../thesis/052_MD/0522_MD_trajectories").resolve()

template_file = Path("./templates/in.lammps_complex-gen").resolve()

seed_csv = Path("../data/simulation-seeds.csv").resolve()
# Change this if your CSV is somewhere else, for example:
# seed_csv = Path("../thesis/052_MD/simulation-seeds.csv").resolve()

if not md_root.exists():
    raise FileNotFoundError(f"MD trajectories root not found: {md_root}")

if not template_file.exists():
    raise FileNotFoundError(f"LAMMPS template not found: {template_file}")

if not seed_csv.exists():
    raise FileNotFoundError(f"Seed CSV not found: {seed_csv}")

template_text = template_file.read_text()
seeds_df = pd.read_csv(seed_csv)

print(f"MD root: {md_root}")
print(f"Template: {template_file}")
print(f"Seed CSV: {seed_csv}")
print(f"Seed rows: {len(seeds_df)}")

In [ ]:
# =========================================================
# Host offsets
# =========================================================

MOF_OFFSETS = {
    "MOF-545": {"MOF_A": 9, "MOF_B": 147, "MOF_C": 638, "MOF_D": 3, "MOF_E": 0},
    "NU-902":  {"MOF_A": 9, "MOF_B": 57,  "MOF_C": 234, "MOF_D": 3, "MOF_E": 0},
    "PCN-223": {"MOF_A": 8, "MOF_B": 58,  "MOF_C": 252, "MOF_D": 5, "MOF_E": 0},
    "PCN-225": {"MOF_A": 9, "MOF_B": 104, "MOF_C": 457, "MOF_D": 3, "MOF_E": 0},
}

# Fallback per-atom topology space.
# Keep these unless you know the exact max values from a LAMMPS log.
DEFAULT_BA = 6
DEFAULT_AA = 12
DEFAULT_DA = 24
DEFAULT_IA = 0


TYPES_PATTERNS = {
    "AT": re.compile(r"^\s*(\d+)\s+atom types\s*$", re.IGNORECASE),
    "BT": re.compile(r"^\s*(\d+)\s+bond types\s*$", re.IGNORECASE),
    "AnT": re.compile(r"^\s*(\d+)\s+angle types\s*$", re.IGNORECASE),
    "DT": re.compile(r"^\s*(\d+)\s+dihedral types\s*$", re.IGNORECASE),
    "IT": re.compile(r"^\s*(\d+)\s+improper types\s*$", re.IGNORECASE),
}


def parse_data_types(data_file):
    """
    Parse atom/bond/angle/dihedral/improper type counts from a LAMMPS data file.
    """
    text = Path(data_file).read_text(errors="ignore")
    out = {"AT": None, "BT": None, "AnT": None, "DT": None, "IT": 0}

    for line in text.splitlines():
        for key, pattern in TYPES_PATTERNS.items():
            match = pattern.match(line)
            if match:
                out[key] = int(match.group(1))

    missing = [key for key, value in out.items() if value is None]
    if missing:
        raise ValueError(f"Missing type counts in {data_file}: {missing}")

    return out


def apply_replacements(template_text, mapping):
    """
    Replace placeholders in the LAMMPS template.
    Handles both raw placeholders like GUEST, HOST-NAME, RAND_VEL
    and bracket placeholders like <GUEST>.
    """
    text = template_text

    # Replace longer/specific keys first to avoid partial replacements.
    for key in sorted(mapping, key=len, reverse=True):
        value = str(mapping[key])
        text = text.replace(f"<{key}>", value)
        text = text.replace(key, value)

    return text


def get_seed_row(seeds_df, host_name, guest_name):
    """
    Find the row for this host + guest abbreviation.
    Example: host_name='NU-902', guest_name='NAP'
    """
    match = seeds_df[
        (seeds_df["host"] == host_name) &
        (seeds_df["guest_abbr"] == guest_name)
    ]

    if match.empty:
        return None

    return match.iloc[0]


def find_host_data_file(host_dir, host_name):
    """
    Find host data file inside the host folder.
    Example:
    data.NU-902_cond_dftb_geo-opt_EQeq
    """
    candidates = sorted(host_dir.glob(f"data.{host_name}*"))

    if not candidates:
        return None

    return candidates[0]


def find_guest_centered_data_file(guest_dir, guest_name):
    """
    Find centered guest data file.
    Example:
    data.NAP_centered
    """
    candidates = [
        guest_dir / f"data.{guest_name}_centered",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    # fallback, in case the naming differs slightly
    matches = sorted(guest_dir.glob("data.*_centered"))
    if matches:
        return matches[0]

    return None

In [ ]:
# =========================================================
# Generate LAMMPS input files
# =========================================================

runs = [
    ("300K", 300, "md_seed1_300K"),
    ("300K", 300, "md_seed2_300K"),
    ("300K", 300, "md_seed3_300K"),
    ("600K", 600, "md_seed_600K"),
    ("900K", 900, "md_seed_900K"),
    ("1200K", 1200, "md_seed_1200K"),
    ("1500K", 1500, "md_seed_1500K"),
]

host_dirs = sorted([p for p in md_root.iterdir() if p.is_dir()])

print(f"Found {len(host_dirs)} host folders")

for host_dir in host_dirs:
    host_name = host_dir.name

    if host_name not in MOF_OFFSETS:
        print(f"\n[skip host] {host_name}: no MOF offsets defined")
        continue

    host_data_file = find_host_data_file(host_dir, host_name)

    if host_data_file is None:
        print(f"\n[skip host] {host_name}: no host data file found")
        continue

    print(f"\n==============================")
    print(f"[HOST] {host_name}")
    print(f"Host data: {host_data_file.name}")

    guest_dirs = sorted([p for p in host_dir.iterdir() if p.is_dir()])

    print(f"Found {len(guest_dirs)} guest folders for {host_name}")

    for guest_dir in guest_dirs:
        guest_name = guest_dir.name

        seed_row = get_seed_row(seeds_df, host_name, guest_name)

        if seed_row is None:
            print(f"  [skip] {host_name} / {guest_name}: no seed row in CSV")
            continue

        guest_data_file = find_guest_centered_data_file(guest_dir, guest_name)

        if guest_data_file is None:
            print(f"  [skip] {host_name} / {guest_name}: no centered guest data file")
            continue

        try:
            guest_types = parse_data_types(guest_data_file)
        except Exception as e:
            print(f"  [skip] {host_name} / {guest_name}: failed to parse guest data types: {e}")
            continue

        base_mapping = {
            "HOST-NAME": host_name,
            "GUEST": guest_name,
            "MOF-DATA": host_data_file.name,

            # Guest type offsets from guest data file
            "AT": guest_types["AT"],
            "BT": guest_types["BT"],
            "AnT": guest_types["AnT"],
            "DT": guest_types["DT"],
            "IT": guest_types["IT"],

            # Per-atom topology reservation
            "BA": DEFAULT_BA,
            "AA": DEFAULT_AA,
            "DA": DEFAULT_DA,
            "IA": DEFAULT_IA,

            # Host offsets
            **MOF_OFFSETS[host_name],
        }

        print(f"\n  [guest] {guest_name}")
        print(f"  Guest data: {guest_data_file.name}")
        print(f"  Types: AT={guest_types['AT']} BT={guest_types['BT']} AnT={guest_types['AnT']} DT={guest_types['DT']} IT={guest_types['IT']}")

        for label, temp, seed_col in runs:
            seed = int(seed_row[seed_col])

            run_dir = guest_dir / f"{temp}K_seed_{seed}"
            run_dir.mkdir(parents=True, exist_ok=True)

            mapping = {
                **base_mapping,
                "TEST_TEMP": temp,
                "RAND_VEL": seed,
            }

            input_text = apply_replacements(template_text, mapping)

            output_input = run_dir / "in.lammps_complex-gen"
            output_input.write_text(input_text)

            print(f"    [OK] {output_input.relative_to(md_root)}")

All MD input files are now in place. Generate SLURM jobs to run the trajectories
on the HPC; the entire `052_MD/0522_MD_trajectories/` folder is self-contained and
can be copied to the cluster as-is. Once the trajectories finish, the
configuration-sampling stage is complete — energetic screening continues in
[`energetic-data-analysis.ipynb`](./energetic-data-analysis.ipynb).